In [0]:
from pyspark.sql import functions as F

from src.config import config
from src.ingestion import (
    create_empty_table,
    ingest_bronze_dim_table,
)
from src.transformation import (
    standardize_customers,
    standardize_franchises,
    standardize_transactions,
)
from src.quality import (
    check_global_quality,
    validate_customers,
    validate_franchises,
    split_valid_invalid,
)
from src.pipeline import AUDIT_SCHEMA

In [0]:
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {config.catalog}.{config.schema}"
)

#### Customer table

In [0]:
ingest_bronze_dim_table(spark, config.source_customers, config.bronze_customers, config.source_customers)
customers_std = standardize_customers(spark.table(config.bronze_customers))
customers_dedup, customers_quality = check_global_quality(customers_std, "customerID")
customers_checked = validate_customers(customers_dedup)
customers_valid, customers_invalid = split_valid_invalid(customers_checked)

(
    customers_valid
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(config.silver_customers)
)

print("Valid customers:", customers_valid.count())
print("Invalid customers:", customers_invalid.count())

#### Franchise Table

In [0]:
ingest_bronze_dim_table(spark, config.source_franchises, config.bronze_franchises, config.source_franchises)
franchises_std = standardize_franchises(spark.table(config.bronze_franchises))
franchises_dedup, franchises_quality = check_global_quality(franchises_std, "franchiseID")
franchises_checked = validate_franchises(franchises_dedup)
franchises_valid, franchises_invalid = split_valid_invalid(franchises_checked)

(
    franchises_valid
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(config.silver_franchises)
)

print("Valid franchises:", franchises_valid.count())
print("Invalid franchises:", franchises_invalid.count())

#### Transaction table 

In [0]:
bronze_template = (
    spark.table(config.source_transactions)
    .withColumn("_batch_id", F.lit(None).cast("string"))
    .withColumn("_run_id", F.lit(None).cast("int"))
    .withColumn("_ingested_at", F.lit(None).cast("timestamp"))
    .withColumn("_source", F.lit(None).cast("string"))
)
create_empty_table(bronze_template, config.bronze_transactions)

silver_template = standardize_transactions(bronze_template)
create_empty_table(silver_template, config.silver_transactions)

rejects_template = validate_transactions(
    silver_template,
    spark.table(config.silver_customers),
    spark.table(config.silver_franchises),
)
create_empty_table(rejects_template, config.silver_invalid_transactions)

audit_df = spark.createDataFrame([], AUDIT_SCHEMA)
create_empty_table(audit_df, config.pipeline_audit)

#### Initialization check

In [0]:
for table_name in [
    config.bronze_customers, config.bronze_franchises,
    config.silver_customers, config.silver_franchises,
    config.bronze_transactions, config.silver_transactions,
    config.silver_invalid_transactions, config.pipeline_audit,
]:
    print(f"{table_name}: {spark.table(table_name).count()} rows")